# CSIS3764 Exam Q3 — End-of-Year 2023
## Cognac Dataset — Unsupervised Learning

## 3.1 — Import dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

cognac = pd.read_csv('cognac.csv')
print(cognac.head())

## 3.2 — Inspect the data

In [ ]:
# Last 10 records
print(cognac.tail(10))

# Concise summary
cognac.info()

## 3.3 — k-Means clustering

In [ ]:
# Pre-processing
print('Checking for missing values:')
print(cognac.isnull().sum())

# Drop non-numeric columns if any
cognac_numeric = cognac.select_dtypes(include=[np.number])
print('Numeric columns:', cognac_numeric.columns.tolist())

# Scale features
scaler = StandardScaler()
cognac_scaled = scaler.fit_transform(cognac_numeric)
print('Scaled shape:', cognac_scaled.shape)

In [ ]:
# Elbow curve
k_range = range(2, 11)
inertia = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(cognac_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(cognac_scaled, labels))

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o', color='blue')
plt.title('Elbow Curve')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.xticks(k_range)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(k_range, sil_scores, marker='o', color='orange')
plt.title('Silhouette Score per k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.xticks(k_range)
plt.tight_layout()
plt.show()

best_k = list(k_range)[sil_scores.index(max(sil_scores))]
print(f'Best k: {best_k} (silhouette = {max(sil_scores):.4f})')

In [ ]:
# Train final model
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cognac['Cluster'] = kmeans.fit_predict(cognac_scaled)

print('Cluster labels:')
print(cognac['Cluster'].values)
print('\nCluster counts:')
print(cognac['Cluster'].value_counts())

## 3.4 — PCA + scatter plot

In [ ]:
pca = PCA(n_components=2)
cognac_pca = pca.fit_transform(cognac_scaled)

print('Original dimensions:', cognac_scaled.shape)
print('PCA dimensions:     ', cognac_pca.shape)

print('\nPrincipal components:')
print(pca.components_)

print('\nExplained variance:')
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {var:.4f} ({var*100:.2f}%)')
print(f'  Total: {pca.explained_variance_ratio_.sum()*100:.2f}%')

pca_df = pd.DataFrame(cognac_pca, columns=['PC1', 'PC2'])
pca_df['Cluster'] = cognac['Cluster']

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Cluster',
                palette='tab10', s=60, alpha=0.8)
plt.title('PCA Scatter Plot — Cognac Clusters')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.tight_layout()
plt.show()

## 3.5 — Evaluation and discussion

In [ ]:
print(f"""
=== PCA Effectiveness and Cluster Evaluation ===

Total variance retained: {pca.explained_variance_ratio_.sum()*100:.2f}%

PCA Effectiveness:
PCA reduced the dataset from {cognac_scaled.shape[1]} dimensions to 2 dimensions
while retaining {pca.explained_variance_ratio_.sum()*100:.2f}% of the total variance.
{'This is excellent — the 2D representation is a reliable reflection of the original data.' if pca.explained_variance_ratio_.sum() >= 0.95 else 'This is acceptable, though some information was lost in the reduction.'}

Cluster Evaluation:
If the clusters are visibly separated in the scatter plot with minimal
overlap, the k-means algorithm has successfully identified distinct
groupings in the cognac content data. Overlapping clusters would suggest
the natural groupings are not well separated in 2D PCA space, though
they may still be distinguishable in the original higher-dimensional space.

The optimal k was determined as {best_k} based on the silhouette score.
If the scatter plot shows {best_k} clearly separated groups this confirms
the chosen k is appropriate for this dataset.
""")